# Bootstrap Error Estimation for ppxf Velocity Dispersion

Empirical error bars via hybrid wild bootstrap with local residual scaling.
Compares bootstrap errors to ppxf formal errors across polynomial degrees
and template libraries (FSPS, EMILES, XSL).

**Method:** For each polynomial degree and template library:
1. Take the best-fit spectrum as the model
2. Compute residuals and their local (rolling-window) scatter
3. Generate N perturbed spectra via Rademacher wild bootstrap (sign-flip × local-scaled residuals)
4. Re-fit each with ppxf → distribution of V, sigma
5. Bootstrap error = std of the distribution

**Target:** AGEL0206 deflector galaxy, z = 0.675  
**Instrument:** Keck/KCWI medium slicer, R~4000

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')
from scripts.bootstrap_ppxf import (
    setup_ppxf_inputs, run_bootstrap, run_regularization_scan,
    compute_local_residual_scaling
)

# Plotting defaults
plt.rcParams['figure.facecolor'] = 'white'
plt.rc('font', family='serif', size=14)
plt.rc('axes', linewidth=1.5, labelsize=16)
plt.rc('xtick', labelsize=14, direction='in')
plt.rc('ytick', labelsize=14, direction='in')

ifu_file = '../Nov17_2025_DESJ0206_RL_combined_icubes_wcs.fits'
results_dir = '../results'

## 2. Quick test (N=50, 4 degrees)

In [ ]:
# Quick test to verify the pipeline works (~10 seconds)
test_results = run_bootstrap(
    ifu_file=ifu_file, sps_name='fsps', results_dir=results_dir,
    degrees=np.array([4, 10, 16, 20]),
    n_bootstrap=50, seed=42, save=False,
)
print("\nTest bootstrap sigma errors:", test_results['sigma_boot_err'])
print("Test bootstrap V errors:", test_results['V_boot_err'])

## 3. Full bootstrap run (all 3 templates, N=500)

In [ ]:
# Full run for all three template libraries (~12 min each, ~36 min total)
all_results = {}
for sps in ['fsps', 'emiles', 'xsl']:
    print(f"\n{'=' * 60}")
    print(f"Running bootstrap for {sps}...")
    print(f"{'=' * 60}")
    all_results[sps] = run_bootstrap(
        ifu_file=ifu_file, sps_name=sps, results_dir=results_dir,
        n_bootstrap=500, seed=42, save=True,
    )

## 4. Residual diagnostics

In [ ]:
# Visualize residuals and local scaling for one template/degree
sps_show = 'fsps'
deg_show = 16

# Load saved results
saved = np.load(f'{results_dir}/ppxf_integrated_spectrum_results_{sps_show}.npz', allow_pickle=True)
galaxy = saved['galaxy']
lam_rest = saved['lam_gal_rest']
bf = saved['best_fit'][deg_show]
resid = galaxy - bf

scale = compute_local_residual_scaling(resid, window=75)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Top: spectrum and best fit
axes[0].step(lam_rest, galaxy, 'k', lw=0.8, label='Data')
axes[0].step(lam_rest, bf, 'r', lw=0.8, label=f'Best fit (deg={deg_show})')
axes[0].set_ylabel('Normalized Flux')
axes[0].legend()
axes[0].set_title(f'Residual diagnostics — {sps_show} templates, degree={deg_show}')

# Middle: residuals
axes[1].scatter(lam_rest, resid, s=2, alpha=0.5, c='b')
axes[1].axhline(0, c='k', ls='--', lw=0.5)
axes[1].set_ylabel('Residual')

# Bottom: local scaling factor
axes[2].plot(lam_rest, scale, 'purple', lw=1.5)
axes[2].axhline(1, c='k', ls='--', lw=0.5)
axes[2].set_ylabel('Local/Global std ratio')
axes[2].set_xlabel(r'Rest Wavelength (\AA)')
axes[2].set_ylim(0, 3)

plt.tight_layout()
plt.show()

## 5. Bootstrap vs formal errors

In [ ]:
# Compare bootstrap errors to ppxf formal errors across all degrees
colors = {'fsps': 'C0', 'emiles': 'C1', 'xsl': 'C2'}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for sps, res in all_results.items():
    # Load formal errors from original results
    orig = np.load(f'{results_dir}/ppxf_integrated_spectrum_results_{sps}.npz')
    degs = res['degrees']
    
    # Find matching indices in the original results
    orig_degs = orig['degrees']
    idx = [np.where(orig_degs == d)[0][0] for d in degs]
    
    axes[0].plot(degs, orig['error_vdis'][idx], '--', color=colors[sps],
                 alpha=0.5, label=f'{sps} formal')
    axes[0].plot(degs, res['sigma_boot_err'], '-o', color=colors[sps],
                 markersize=4, label=f'{sps} bootstrap')
    
    axes[1].plot(degs, orig['error_vel'][idx], '--', color=colors[sps], alpha=0.5)
    axes[1].plot(degs, res['V_boot_err'], '-o', color=colors[sps], markersize=4)

axes[0].set_xlabel('Additive Polynomial Degree')
axes[0].set_ylabel(r'$\delta\sigma$ (km/s)')
axes[0].set_title(r'Velocity Dispersion Error: Bootstrap vs Formal')
axes[0].legend(fontsize=9, ncol=2)
axes[0].grid(alpha=0.3)

axes[1].set_xlabel('Additive Polynomial Degree')
axes[1].set_ylabel(r'$\delta V$ (km/s)')
axes[1].set_title(r'Mean Velocity Error: Bootstrap vs Formal')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Bootstrap distributions (histograms)

In [ ]:
# Histograms of sigma for selected degrees, one row per template
deg_show = [4, 10, 16, 20]
templates = ['fsps', 'emiles', 'xsl']

fig, axes = plt.subplots(len(templates), len(deg_show), figsize=(16, 3*len(templates)))

for i, sps in enumerate(templates):
    res = all_results[sps]
    for j, deg in enumerate(deg_show):
        ax = axes[i, j]
        deg_idx = np.where(res['degrees'] == deg)[0]
        if len(deg_idx) == 0:
            ax.text(0.5, 0.5, f'deg={deg}\nnot available', transform=ax.transAxes, ha='center')
            continue
        deg_idx = deg_idx[0]
        
        samples = res['sigma_bootstrap'][deg_idx]
        valid = samples[np.isfinite(samples)]
        
        ax.hist(valid, bins=30, color=colors[sps], alpha=0.6, edgecolor=colors[sps])
        ax.axvline(res['sigma_original'][deg_idx], color='k', ls='-', lw=2, label='Original')
        ax.axvline(np.nanmedian(samples), color='r', ls='--', lw=1.5, label='Boot median')
        
        if i == 0:
            ax.set_title(f'degree={deg}')
        if j == 0:
            ax.set_ylabel(f'{sps}\nCount')
        if i == len(templates) - 1:
            ax.set_xlabel(r'$\sigma$ (km/s)')
        if i == 0 and j == 0:
            ax.legend(fontsize=8)

plt.suptitle(r'Bootstrap $\sigma$ distributions', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Sigma stability with bootstrap error bands

In [ ]:
# Key summary: sigma vs degree with shaded 1-sigma bootstrap bands
fig, ax = plt.subplots(figsize=(12, 7))

for sps in ['fsps', 'emiles', 'xsl']:
    res = all_results[sps]
    degs = res['degrees']
    sigma = res['sigma_original']
    err = res['sigma_boot_err']
    
    ax.fill_between(degs, sigma - err, sigma + err, alpha=0.2, color=colors[sps])
    ax.plot(degs, sigma, '-o', color=colors[sps], markersize=5, label=sps)

ax.set_xlabel('Additive Polynomial Degree')
ax.set_ylabel(r'$\sigma$ (km/s)')
ax.set_title(r'Velocity Dispersion vs Polynomial Degree (bootstrap 1$\sigma$ bands)')
ax.legend(fontsize=12)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{results_dir}/figures/sigma_vs_degree_bootstrap.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Regularization analysis

In [ ]:
# Regularization scan for degree=16, one template
ppxf_inputs = setup_ppxf_inputs(ifu_file, sps_name='fsps')
regul_results = run_regularization_scan(ppxf_inputs, degree=16)

In [ ]:
# Plot regularization results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

rv = regul_results['regul_values']

# chi2 vs regul
axes[0].semilogx(rv, regul_results['chi2'], 'b-o', markersize=3)
axes[0].axhline(regul_results['chi2_target'], color='r', ls='--', label=f'Target chi2')
axes[0].axvline(regul_results['optimal_regul'], color='g', ls=':', label=f'Optimal regul={regul_results["optimal_regul"]:.1f}')
axes[0].set_xlabel('regul')
axes[0].set_ylabel(r'$\chi^2$/DOF')
axes[0].set_title(r'$\chi^2$ vs regularization')
axes[0].legend()
axes[0].grid(alpha=0.3)

# sigma vs regul
axes[1].semilogx(rv, regul_results['sigma'], 'r-o', markersize=3)
axes[1].axvline(regul_results['optimal_regul'], color='g', ls=':')
axes[1].set_xlabel('regul')
axes[1].set_ylabel(r'$\sigma$ (km/s)')
axes[1].set_title(r'$\sigma$ vs regularization')
axes[1].grid(alpha=0.3)

# V vs regul
axes[2].semilogx(rv, regul_results['V'], 'g-o', markersize=3)
axes[2].axvline(regul_results['optimal_regul'], color='g', ls=':')
axes[2].set_xlabel('regul')
axes[2].set_ylabel('V (km/s)')
axes[2].set_title('V vs regularization')
axes[2].grid(alpha=0.3)

plt.suptitle('Regularization scan (degree=16, fsps)', fontsize=14)
plt.tight_layout()
plt.show()

## 9. Summary

In [ ]:
# Summary table at degree=16 for all templates
deg_report = 16
print(f"{'Template':>8} {'sigma':>8} {'boot_err':>9} {'formal_err':>10} {'V':>8} {'V_boot_err':>10}")
print('-' * 60)
for sps in ['fsps', 'emiles', 'xsl']:
    res = all_results[sps]
    idx = np.where(res['degrees'] == deg_report)[0]
    if len(idx) == 0:
        continue
    idx = idx[0]
    orig = np.load(f'{results_dir}/ppxf_integrated_spectrum_results_{sps}.npz')
    orig_idx = np.where(orig['degrees'] == deg_report)[0][0]
    print(f"{sps:>8} {res['sigma_original'][idx]:8.1f} {res['sigma_boot_err'][idx]:9.1f} "
          f"{orig['error_vdis'][orig_idx]:10.1f} {res['V_original'][idx]:8.1f} {res['V_boot_err'][idx]:10.1f}")